# OLS: (CPI - c_diff) on lagged (xi * oil)

This notebook estimates:

- Regressand: `cpi_pct_change - c_diff`
- Main regressor: `xi_x_oil(t-1)`
- Optional additional regressor: contemporaneous `xi_x_oil(t)`
- Optional additional regressors: lags of `(cpi_pct_change - c_diff)`
- Optional country fixed effects


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

import regression


In [ ]:
ROOT = Path.cwd().resolve().parent

xi_path = ROOT / "data/processed/xi/regular/xi_by_country_year.parquet"
c_path = ROOT / "data/processed/c/regular/c_by_country_year.parquet"
c_diff_path = ROOT / "data/processed/c_diff/regular/c_diff_by_country_year.parquet"
cpi_freq = "A"  # this spec uses annual CPI
cpi_path = ROOT / "data/processed/cpi/annual_cpi.csv"
oil_path = ROOT / "data/processed/oil/annual_oil.csv"

# Model controls
xi_oil_lag = 1  # t-1 as requested
include_xi_oil_t = False  # set True to include contemporaneous xi_x_oil(t)
y_lags = 1  # number of lags of (cpi - c_diff)
include_country_fe = True
exclude_argentina = True

print("xi_oil_lag:", xi_oil_lag)
print("include_xi_oil_t:", include_xi_oil_t)
print("y_lags:", y_lags)
print("include_country_fe:", include_country_fe)
print("exclude_argentina:", exclude_argentina)


In [ ]:
xi_df = regression.load_xi(xi_path)
c_df = regression.load_c(c_path)
c_diff_df = regression.load_c_diff(c_diff_path)
cpi_df = regression.load_cpi_pct_change(cpi_path, freq=cpi_freq)
oil_df = regression.load_oil_pct_change(oil_path, freq=cpi_freq)

reg_df = regression.prepare_regression_df(xi_df, c_df, cpi_df, oil_df, c_diff_df=c_diff_df)
if exclude_argentina:
    reg_df = regression.exclude_countries(reg_df, ["ARG"])

reg_df = reg_df.sort_values(["country", "year"]).reset_index(drop=True)
reg_df["y"] = reg_df["cpi_pct_change"] - reg_df["c_diff"]
reg_df["xi_x_oil_lag"] = reg_df.groupby("country")["xi_x_oil"].shift(xi_oil_lag)

lag_cols = []
for lag in range(1, y_lags + 1):
    col = f"y_lag{lag}"
    reg_df[col] = reg_df.groupby("country")["y"].shift(lag)
    lag_cols.append(col)

print("Rows (pre-design):", len(reg_df))
print("Countries:", reg_df["country"].nunique())
print("Years:", int(reg_df["year"].min()), "-", int(reg_df["year"].max()))


In [ ]:
x_cols = ["xi_x_oil_lag"]
if include_xi_oil_t:
    x_cols.append("xi_x_oil")
x_cols += lag_cols
x_base = reg_df[x_cols].copy()

fe_cols = []
if include_country_fe:
    fe = pd.get_dummies(reg_df["country"], prefix="country_fe", drop_first=True, dtype=float)
    fe_cols = fe.columns.tolist()
    x_base = pd.concat([x_base, fe], axis=1)

design = pd.concat([reg_df[["country", "year", "y"]], x_base], axis=1).dropna().reset_index(drop=True)
if design.empty:
    raise ValueError("No rows left for OLS after lagging/dropna. Reduce lags or inspect missing data.")

y = design["y"]
x = sm.add_constant(design[x_cols + fe_cols], has_constant="add")
model = sm.OLS(y, x, missing="raise").fit(cov_type="HC3")

ci = model.conf_int()
coef_df = pd.DataFrame(
    {
        "term": model.params.index,
        "coef": model.params.values,
        "std_err_hc3": model.bse.values,
        "t": model.tvalues.values,
        "p_value": model.pvalues.values,
        "ci_low_95": ci.iloc[:, 0].values,
        "ci_high_95": ci.iloc[:, 1].values,
    }
)

print("Rows used in OLS:", int(model.nobs))
coef_df


In [ ]:
print(model.summary())


In [ ]:
x_scatter = design["xi_x_oil_lag"].to_numpy(dtype=float)
y_scatter = design["y"].to_numpy(dtype=float)

x_line = np.linspace(np.nanmin(x_scatter), np.nanmax(x_scatter), 200)
y_line = model.params["const"] + model.params["xi_x_oil_lag"] * x_line

# Add average lag contributions for visual line anchoring.
for col in lag_cols:
    if col in model.params.index:
        y_line += model.params[col] * float(design[col].mean())

plt.figure(figsize=(9, 6))
plt.scatter(x_scatter, y_scatter, alpha=0.35, s=18)
plt.plot(x_line, y_line, linewidth=2)
plt.axhline(0.0, linewidth=1)
plt.axvline(0.0, linewidth=1)
plt.xlabel(f"xi * oil pct change (t-{xi_oil_lag})")
plt.ylabel("cpi_pct_change - c_diff")
fe_label = "country FE" if include_country_fe else "no country FE"
x_label = "xi*oil(t-1) + xi*oil(t)" if include_xi_oil_t else "xi*oil(t-1)"
plt.title(f"y = CPI - c_diff; x = {x_label} ({fe_label}, y_lags={y_lags})")
plt.tight_layout()
plt.show()
